# Grafisk brukergrensesnitt med tkinter

Det finnes [flere pakker](https://wiki.python.org/moin/GuiProgramming) for å lage vindusbaserte programmer i Python. Vi bruker **tkinter**, fordi den

* er enkel å bruke og krever lite kode
* følger med Python (ingen installasjon)
* virker på Windows, Mac og Linux
* er moden og stabil, med mange utvidelser

> **Merk:** tkinter-programmer åpner et eget vindu. Lukk vinduet for å gå videre i notebooken. På en server uten skjerm (for eksempel Google Colab) virker ikke tkinter – kopier da koden til en `.py`-fil og kjør den lokalt.

## 1 Rammeverket

Alt i tkinter er **widgets**: `Label`, `Entry`, `Button`, `Checkbutton` osv.
Hvert program har et hovedvindu (`root`) og avsluttes med `mainloop()`, som venter på hendelser (klikk, tastetrykk) til vinduet lukkes.

In [ ]:
from tkinter import *

root = Tk()
Label(root, text='Hallo!').pack()
root.mainloop()

`Label` lager en tekst i `root`-vinduet, og `pack()` legger den ut på skjermen.

## 2 Inndata, knapp og utdata

Et program som leser inn et navn og bruker det i en hilsen:

In [ ]:
from tkinter import *

root = Tk()

def skriv_ut():
    hilsen.configure(text='Hallo ' + navn.get() + '!')

navn = Entry(root)
navn.insert(0, 'Hva heter du? ')
navn.pack()

Button(root, text='Les navn', command=skriv_ut).pack()

hilsen = Label(root)
hilsen.pack()

root.mainloop()

* `Entry` er et **inndatafelt**. Vi trenger en referanse (`navn`) til feltet senere, så vi kan ikke lage og legge det ut i samme setning – `pack()` returnerer `None`. `insert` legger inn en forklarende tekst.
* `Button` er en **knapp**. `command=` kobler knappen til en funksjon. Legg merke til at det står `skriv_ut` *uten* parenteser: vi gir knappen funksjonen, vi kaller den ikke.
* `Label` er **utdatafeltet**, først med tom tekst.
* Når vi klikker, kalles `skriv_ut`. Den henter teksten med `get()` og endrer utdatafeltet med `configure()`.

## 3 Utseende og grid

Programmet over er ikke særlig pent. Her endrer vi tittellinja, standardskrifta og farger, legger inn et bilde og bruker `grid()` i stedet for `pack()`.

Med `grid()` plasserer vi widgetene i et **rutenett** med rad og kolonne. Rutenettet blir ikke større enn plassen widgetene trenger.

| Valg | Betydning |
|---|---|
| `row`, `column` | rad og kolonne |
| `rowspan`, `columnspan` | gå over flere rader/kolonner |
| `padx`, `pady` | luft utenfor (i `grid`) eller inni (i widgeten) |
| `width` | bredde (i tegn for tekst-widgets) |
| `fg`, `bg` | tekstfarge og bakgrunnsfarge |
| `sticky` | strekk/juster i ruta: `N`, `S`, `E`, `W` |

Bildet krever biblioteket Pillow (`pip install Pillow`). Har du ikke et bilde, tegner vi en enkel logo selv:

In [ ]:
from tkinter import *
from tkinter import font
from PIL import Image, ImageTk, ImageDraw   # pip install Pillow

root = Tk()

def skriv_ut():
    hilsen.configure(text='Hallo ' + navn.get() + '!')

# Endre standardskrifta (gjelder ikke Entry)
standard = font.nametofont('TkDefaultFont')
standard.configure(family='Arial', size=20)

root.title('Inndata/Utdata')

# Bilde: bytt ut med Image.open('ditt_bilde.jpg') hvis du har et
bilde = Image.new('RGB', (150, 150), '#3877ac')
ImageDraw.Draw(bilde).text((40, 65), 'tkinter', fill='white')
bilde = bilde.resize((150, 150), Image.Resampling.LANCZOS)
bilde = ImageTk.PhotoImage(bilde)
Label(root, image=bilde).grid(row=0, column=0, padx=5, pady=5)

# En ramme for høyre del
ramme = LabelFrame(root, padx=10, pady=10)
ramme.grid(row=0, column=1, padx=5, pady=15)

navn = Entry(ramme, width=15, font=('Arial', 20))
navn.insert(0, 'Hva heter du? ')
navn.grid(row=0, column=0, padx=5, pady=13)

Button(ramme, text='Les navn', command=skriv_ut, width=10).grid(row=0, column=1, padx=5)

hilsen = Label(ramme, width=25, bg='#3877ac', fg='white', pady=3)
hilsen.grid(row=1, column=0, columnspan=2, pady=13)

root.mainloop()

## 4 Spinbox og unntakshåndtering

Tall kan leses inn med `Entry`, men en `Spinbox` gjør det lettere å velge tall. Den hindrer likevel ikke brukeren i å skrive noe annet, så vi fanger feil med `try` og tømmer feltene.
Selv om vi lar `Spinbox` gå fra 0 til 10, kan brukeren skrive inn større tall. Her tillater vi desimaltall, men runder av svaret.

In [ ]:
from tkinter import *
from tkinter import font

root = Tk()

def gange():
    try:
        svar = float(tall_1.get()) * float(tall_2.get())
    except ValueError:
        tall_1.delete(0, END)
        tall_2.delete(0, END)
        resultat.configure(text='')
    else:
        resultat.configure(text=f'{svar:.0f}')

root.title('Spinbox')
font.nametofont('TkDefaultFont').configure(family='Arial', size=20)

tall_1 = Spinbox(root, from_=0, to=10, increment=1, width=5, font=('Arial', 20))
tall_1.grid(row=0, column=0, padx=10, pady=10)
Label(root, text='*').grid(row=0, column=1, padx=10, pady=10)
tall_2 = Spinbox(root, from_=0, to=10, increment=1, width=5, font=('Arial', 20))
tall_2.grid(row=0, column=2, padx=10, pady=10)
Button(root, text='=', command=gange, width=5).grid(row=0, column=3, padx=10, pady=10)
resultat = Label(root, width=5)
resultat.grid(row=0, column=4, padx=10, pady=10)

root.mainloop()

`from_` har understrek fordi `from` er et reservert ord i Python.

## 5 Radioknapper

Radioknapper lar brukeren velge **ett** av flere alternativer.

* Vi trenger en tkinter-variabel (`IntVar`, `DoubleVar`, `StringVar` eller `BooleanVar`) som holder verdien til den valgte knappen. Alle knappene i en gruppe bruker **samme** variabel (`variable=`).
* Hver knapp har sin egen verdi (`value=`).
* `command=` er funksjonen som kalles når en knapp velges.
* `anchor=W` (vest) venstrejusterer teksten. Da må `width` være satt.

`enumerate(farger)` gir både indeksen og verdien til hvert element i listen:

In [ ]:
farger = ['Rød', 'Blå', 'Grønn']
for indeks, farge in enumerate(farger):
    print(indeks, farge)

In [ ]:
from tkinter import *
from tkinter import font

root = Tk()
farger = ['Rød', 'Blå', 'Grønn']
fargekoder = ['red', 'blue', 'green']

def velg_farge():
    i = farge_var.get()
    valg.configure(text=farger[i], fg='white', bg=fargekoder[i])

root.title('Radioknapper')
font.nametofont('TkDefaultFont').configure(family='Arial', size=15)

Label(root, text='Velg farge:', bg='darkgrey', width=20, padx=5, pady=5) \
    .grid(row=0, column=0, padx=5, pady=5)

farge_var = IntVar()
for indeks, farge in enumerate(farger):
    rb = Radiobutton(root, text=farge, variable=farge_var, value=indeks,
                     command=velg_farge, width=10, anchor=W)
    rb.grid(row=indeks + 1, column=0)
farge_var.set(-1)            # ingen valgt fra start

valg = Label(root, text='')
valg.grid(row=4, column=0, padx=5, pady=5, sticky=N+S+E+W)

root.mainloop()

**Avhukingsbokser** (`Checkbutton`) ligner, men der kan brukeren velge flere, og hver boks må ha sin egen variabel. Se for eksempel [pythonbasics.org/tkinter-checkbox](https://pythonbasics.org/tkinter-checkbox/).

## 6 Listbox med rullefelt

* Vi lager en ramme (`LabelFrame`) for `Listbox` og rullefelt (`Scrollbar`).
* `yscrollcommand=rullefelt.set` og `rullefelt.configure(command=lb.yview)` kobler dem sammen begge veier.
* `selectmode=SINGLE` tillater bare én markert linje.
* `bind('<<ListboxSelect>>', funksjon)` kobler *hendelsen* «noe ble valgt» til en funksjon. Funksjonen får et `event`-objekt som argument.
* `curselection()` gir en tuppel med indeksene til det som er markert. Vi tillater bare én, så vi tar `[0]`.

Folketallene er fra rundt 2022.

In [ ]:
from tkinter import *
from tkinter import font

root = Tk()
root.geometry('400x200')
root.title('Listbox')
font.nametofont('TkDefaultFont').configure(family='Arial', size=15)

byer = ['Ålesund', 'Molde', 'Kristiansund', 'Ørsta', 'Volda', 'Ulstein']
innbyggere = [67114, 32350, 24156, 10871, 10753, 8800]

def valgt_by(event):
    i = lb.curselection()[0]
    tekst_ut.configure(text=f'{byer[i]} har {innbyggere[i]} innbyggere.')

ramme = LabelFrame(root)
rullefelt = Scrollbar(ramme, orient=VERTICAL)
rullefelt.pack(side=RIGHT, fill=Y)
ramme.pack(pady=25)

lb = Listbox(ramme, height=3, width=12, yscrollcommand=rullefelt.set, selectmode=SINGLE)
for by in byer:
    lb.insert(END, by)
lb.pack()
rullefelt.configure(command=lb.yview)
lb.bind('<<ListboxSelect>>', valgt_by)

tekst_ut = Label(root, text='')
tekst_ut.pack(pady=10)

root.mainloop()

## 7 Oppgaver

*Variasjon:* Bruk gjerne `pack()` i oppgave 1 og `grid()` i oppgave 2.

### Oppgave 1 – Binære tall med GUI
Lag et grafisk program som gjør om tall fra titallssystemet til binært (oppgaven fra 1.3 og 1.7). Ta gjerne utgangspunkt i terminalversjonen under.

In [ ]:
while True:
    tekst = input('Skriv inn et heltall: ')
    try:
        tall = int(tekst)
    except ValueError:
        if tekst != 'slutt':
            print(f'{tekst} er ikke et heltall')
    else:
        print(f'{tall} i titallssystemet er {tall:b} binært.')
    finally:
        if tekst != 'slutt':
            print('\nVi fortsetter. Skriv "slutt" for å avslutte.')
        else:
            print('\nTakk for nå.')
            break

In [ ]:
# Oppgave 1

### Oppgave 2 – Flytid med GUI
Lag et grafisk program for flytid-oppgaven fra 1.2 (IT2-eksamen våren 2010). Sett gjerne inn et bilde av en fartsmåler. Utgangspunkt:

In [ ]:
import datetime

avstand_i_km = float(input('Oppgi avstand i km: '))
fart_i_knop = float(input('Oppgi fart i knop: '))
flytid = avstand_i_km / (fart_i_knop * 1.852)

tid = str(datetime.timedelta(hours=flytid)).split(':')   # ['t', 'mm', 'ss.ssss']
print(f'Timer: {tid[0]}, minutter: {tid[1]}, sekunder: {tid[2][:2]}')

In [ ]:
# Oppgave 2

### Oppgave 3 – Treningskalkulator (utfordring)
*Basert på oppgave 2 fra IT2-eksamen høsten 2019.*

Lag et program som regner ut kaloriforbruket for en treningsøkt ut fra **aktivitet**, **intensitet** og **varighet**.

* Aktivitet velges i en nedtrekksliste (`ttk.Combobox`, se [Python Tutorial](https://www.pythontutorial.net/tkinter/tkinter-combobox/)):

  | Aktivitet | kcal/time |
  |---|---|
  | Aerobics | 814 |
  | Bordtennis | 236 |
  | Fotball | 510 |
  | Golf | 244 |
  | Jogging | 666 |

* Intensitet velges med tre radioknapper: **Lav** (× 0,8), **Middels** (× 1,0) og **Høy** (× 1,2).
* Varighet oppgis i minutter i et tekstfelt (del på 60 for å få timer).
* Et klikk på knappen *Beregn kaloriforbruk* viser svaret.

```
┌───────────────────────────┐
│ [Velg aktivitet       ▼]  │
│ Velg intensitet:          │
│   ( ) Lav                 │
│   ( ) Middels             │
│   ( ) Høy                 │
│ Varighet i minutter: [__] │
│ [Beregn kaloriforbruk]    │
│ ┌───────────────────────┐ │
│ │ Svar                  │ │
│ └───────────────────────┘ │
└───────────────────────────┘
```

a) Lag et flytdiagram for programmet.
b) Lag programmet. Skill gjerne beregningen ut i en egen funksjon som du kan teste uten GUI:

In [ ]:
AKTIVITETER = {'Aerobics': 814, 'Bordtennis': 236, 'Fotball': 510, 'Golf': 244, 'Jogging': 666}
INTENSITET = {'Lav': 0.8, 'Middels': 1.0, 'Høy': 1.2}

def kaloriforbruk(aktivitet, intensitet, minutter):
    ...

# assert round(kaloriforbruk('Jogging', 'Middels', 60)) == 666
# assert round(kaloriforbruk('Golf', 'Høy', 30)) == 146

In [ ]:
# Oppgave 3b – GUI